# Robustez: validacao cruzada por grupos (GroupKFold)

Os numeros da comparacao principal vem de **um unico split** treino/teste/
validacao (o mesmo do notebook do LSTM). Este notebook estima a
**variabilidade** do desempenho dos tres modelos tabulares (Regressao
Logistica, MLP, Random Forest) com `GroupKFold(n_splits=5)` agrupado por
disco, sobre **todos os discos** do dataset.

Pontos importantes do protocolo:

- **Analise separada do split oficial.** Aqui os 5 folds particionam todos os
  discos; os numeros nao substituem os da comparacao principal (que usa o
  split fixo compartilhado com o LSTM) - servem para mostrar que o ranking
  entre os modelos tabulares e estavel, com media +- desvio padrao por fold.
- **Agrupamento por disco** (`groups=disco_id`): todas as amostras de um mesmo
  disco caem no mesmo fold, evitando vazamento de serie temporal entre treino
  e avaliacao.
- **Apenas AP e AUC.** Sao metricas de ranking, que nao dependem de limiar de
  classificacao - escolher um limiar dentro de cada fold exigiria reservar
  mais um subconjunto de validacao por fold, complicando o protocolo sem
  beneficio para o objetivo (estimar variabilidade do ranking).
- **Hiperparametros fixos**, carregados dos `best_params` salvos nos
  `resultados_*.pkl` oficiais (sem re-busca por fold - o objetivo e medir a
  variabilidade dos dados, nao da busca).
- **Pre-processamento identico ao dos notebooks oficiais:** para LR e MLP, o
  `StandardScaler` e ajustado apenas no treino de cada fold; o RF usa as
  features brutas.
- **O LSTM nao e incluido:** re-treina-lo 5 vezes custaria horas/dias de GPU.
  A analise cobre os modelos tabulares, onde esta a comparacao mais apertada
  (MLP x RF).


In [1]:
# ===== Configuracao =====
import os
import time
import pickle

import numpy as np

import os
import sys

# --- Resolucao robusta de caminhos (independente do diretorio de trabalho) ---
def _encontra_raiz(inicio=None):
    """Sobe na arvore ate achar comum/ssd_utils.py."""
    d = os.path.abspath(inicio or os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, 'comum', 'ssd_utils.py')):
            return d
        pai = os.path.dirname(d)
        if pai == d:
            raise RuntimeError('raiz do projeto (com comum/ssd_utils.py) nao encontrada')
        d = pai

PROJ_ROOT = _encontra_raiz()
COMUM_DIR = os.path.join(PROJ_ROOT, 'comum')
H7_DIR = os.path.join(PROJ_ROOT, 'horizonte_7dias')
H30_DIR = os.path.join(PROJ_ROOT, 'horizonte_30dias')
EXP_DIR = os.path.join(PROJ_ROOT, 'experimentos')
if COMUM_DIR not in sys.path:
    sys.path.insert(0, COMUM_DIR)
# ---------------------------------------------------------------------------
import ssd_utils as ssd

DATA_DIR = COMUM_DIR        # data.pickle / mask.pickle (pasta comum/)
OUTPUT_DIR = EXP_DIR        # saidas desta analise
WINDOW = 90                  # janela oficial escolhida no experimento (AMA_experimento_janela.ipynb)
CONTAMINATION_LEVEL = 7     # horizonte de previsao de falha - igual aos demais notebooks
RANDOM_STATE = 42
N_SPLITS = 5

# Hiperparametros oficiais de cada modelo tabular
best_params = {}
for nome, arq in [('Logistic Regression', 'resultados_logistic_regression.pkl'),
                  ('MLP', 'resultados_mlp.pkl'),
                  ('Random Forest', 'resultados_random_forest.pkl')]:
    path = os.path.join(H7_DIR, arq)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'{path} nao encontrado - execute o notebook de treino correspondente primeiro.')
    with open(path, 'rb') as f:
        best_params[nome] = pickle.load(f)['best_params']
    print(f'{nome}: {best_params[nome]}')


Logistic Regression: {'l1_ratio': 0.5, 'C': 0.01}
MLP: {'learning_rate_init': 0.001, 'hidden_layer_sizes': (128,), 'alpha': 0.01}
Random Forest: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': None}


## 1. Dados, rotulos e features (construidos uma unica vez)

As features de janela deslizante nao dependem do fold - sao construidas uma
unica vez (`WINDOW = 90`) e reutilizadas, indexando as linhas de cada fold
(atencao a memoria: ~1,9M linhas x 120 colunas; nenhuma copia do array
completo e feita dentro do loop).

Como aqui nao usamos o split oficial, filtramos o padding (`mask == 0`) de
todas as linhas de uma vez, mantendo `disco_id` como vetor de grupos para o
`GroupKFold`.


In [2]:
data, mask = ssd.load_dataset(DATA_DIR)
print(f'data: {data.shape}   mask: {mask.shape}')

labels = ssd.create_class_labels(data, mask, CONTAMINATION_LEVEL)

start = time.time()
X_all, disco_id, timestep, feature_cols = ssd.build_windowed_features(data, mask, window=WINDOW)
print(f'X_all: {X_all.shape}   ({len(feature_cols)} features)')
print(f'Tempo de construcao das features: {time.time() - start:.1f}s')

# Remove o padding uma unica vez (equivale ao filtro mask==1 do select_split)
valid = mask.reshape(-1).astype(bool)
X = X_all[valid]
y = labels.reshape(-1)[valid]
groups = disco_id[valid]
del X_all
print(f'Apos remover padding: X {X.shape} | positivos: {int(y.sum())} ({y.mean()*100:.3f}%)')


data: (5343, 360, 24)   mask: (5343, 360)


X_all: (1923480, 120)   (120 features)
Tempo de construcao das features: 21.7s


Apos remover padding: X (1772986, 120) | positivos: 37320 (2.105%)


## 2. Loop de validacao cruzada

Em cada fold, treina nos discos de 4/5 do dataset e avalia no 1/5 restante,
reportando AP e AUC. Os modelos sao re-instanciados a cada fold com os
`best_params` oficiais.


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler


def make_model(nome):
    if nome == 'Logistic Regression':
        return LogisticRegression(
            **best_params[nome],
            penalty='elasticnet', solver='saga',
            class_weight='balanced',
            max_iter=5000, tol=1e-3,
            random_state=RANDOM_STATE,
        ), True   # True = precisa de scaler
    if nome == 'MLP':
        return MLPClassifier(
            **best_params[nome],
            activation='relu', solver='adam', batch_size=512,
            max_iter=100, early_stopping=True, n_iter_no_change=10,
            validation_fraction=0.1, random_state=RANDOM_STATE,
        ), True
    if nome == 'Random Forest':
        return RandomForestClassifier(
            **best_params[nome],
            class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1,
        ), False
    raise ValueError(nome)


MODEL_NAMES = ['Logistic Regression', 'MLP', 'Random Forest']
gkf = GroupKFold(n_splits=N_SPLITS)

resultados = {nome: {'ap': [], 'auc': []} for nome in MODEL_NAMES}

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
    print(f'===== Fold {fold}/{N_SPLITS} - treino {len(tr_idx):,} | avaliacao {len(te_idx):,} '
          f'({y[te_idx].mean()*100:.3f}% positivos) =====')
    for nome in MODEL_NAMES:
        model, needs_scaler = make_model(nome)
        if needs_scaler:
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X[tr_idx])   # ajustado so no treino do fold
            X_te = scaler.transform(X[te_idx])
        else:
            X_tr, X_te = X[tr_idx], X[te_idx]

        t0 = time.time()
        model.fit(X_tr, y[tr_idx])
        fit_time = time.time() - t0

        proba = model.predict_proba(X_te)[:, 1]
        ap = average_precision_score(y[te_idx], proba)
        auc = roc_auc_score(y[te_idx], proba)
        resultados[nome]['ap'].append(ap)
        resultados[nome]['auc'].append(auc)
        print(f'  {nome:<20}: AP={ap:.4f}  AUC={auc:.4f}  (fit: {fit_time/60:.1f} min)')
        del model, X_tr, X_te


===== Fold 1/5 - treino 1,418,388 | avaliacao 354,598 (2.105% positivos) =====


C:\Users\len108\Documents\pv\ErroSSD\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


  Logistic Regression : AP=0.0707  AUC=0.7027  (fit: 1.4 min)


  MLP                 : AP=0.1386  AUC=0.7696  (fit: 7.1 min)


  Random Forest       : AP=0.1694  AUC=0.7926  (fit: 0.9 min)
===== Fold 2/5 - treino 1,418,389 | avaliacao 354,597 (2.105% positivos) =====


...

  Random Forest       : AP=0.1577  AUC=0.7938  (fit: 0.9 min)
===== Fold 5/5 - treino 1,418,389 | avaliacao 354,597 (2.105% positivos) =====


C:\Users\len108\Documents\pv\ErroSSD\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


  Logistic Regression : AP=0.0704  AUC=0.7208  (fit: 1.4 min)


  MLP                 : AP=0.1198  AUC=0.7808  (fit: 12.0 min)


  Random Forest       : AP=0.1491  AUC=0.7820  (fit: 0.9 min)


## 3. Tabela resumo e salvamento

Media +- desvio padrao de AP e AUC sobre os 5 folds, por modelo.


In [4]:
header = f"{'Modelo':<22}{'AP (media+-dp)':>20}{'AUC (media+-dp)':>20}"
print(header)
print('-' * len(header))
resumo = {}
for nome in MODEL_NAMES:
    ap = np.array(resultados[nome]['ap'])
    auc = np.array(resultados[nome]['auc'])
    resumo[nome] = {
        'ap_mean': float(ap.mean()), 'ap_std': float(ap.std()),
        'auc_mean': float(auc.mean()), 'auc_std': float(auc.std()),
        'ap_folds': ap.tolist(), 'auc_folds': auc.tolist(),
    }
    print(f"{nome:<22}{ap.mean():>11.4f} +- {ap.std():.4f}"
          f"{auc.mean():>11.4f} +- {auc.std():.4f}")

ssd.save_results(
    os.path.join(OUTPUT_DIR, 'robustez_groupkfold.pkl'),
    n_splits=N_SPLITS,
    window=WINDOW,
    contamination_level=CONTAMINATION_LEVEL,
    best_params=best_params,
    resumo=resumo,
)


Modelo                      AP (media+-dp)     AUC (media+-dp)
--------------------------------------------------------------
Logistic Regression        0.0707 +- 0.0038     0.7049 +- 0.0083
MLP                        0.1340 +- 0.0149     0.7754 +- 0.0046
Random Forest              0.1639 +- 0.0137     0.7908 +- 0.0045
Resultados salvos em: C:\Users\len108\Documents\pv\ErroSSD\experimentos\robustez_groupkfold.pkl


## 4. Figura: barras com erro

Barras de AP e AUC (media dos 5 folds) com o desvio padrao como barra de erro.


In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

COLORS = {'Logistic Regression': '#2E7D32', 'MLP': '#6A1B9A', 'Random Forest': '#E65100'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle(f'Robustez (GroupKFold, {N_SPLITS} folds por disco) - media +- desvio padrao',
             fontsize=12)

for ax, met, label in zip(axes, ['ap', 'auc'], ['Average Precision', 'ROC-AUC']):
    means = [resumo[n][f'{met}_mean'] for n in MODEL_NAMES]
    stds = [resumo[n][f'{met}_std'] for n in MODEL_NAMES]
    bars = ax.bar(MODEL_NAMES, means, yerr=stds, capsize=5,
                  color=[COLORS[n] for n in MODEL_NAMES],
                  edgecolor='black', linewidth=0.5)
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width() / 2, m + s + 0.01,
                f'{m:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(label)
    ax.set_ylabel(label)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='y', alpha=0.35)

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'robustez_groupkfold.png'), dpi=130, bbox_inches='tight')
print('Figura salva em robustez_groupkfold.png')


Figura salva em robustez_groupkfold.png
